# RAG (Retrieval-Augmented Generation) 系统完整实现

本代码实现了 RAG 的完整流程，分为两大阶段：

## 第一阶段：准备阶段
1. **分片 (Chunking)** - 将长文档切分成小片段
2. **索引 (Indexing)** - 将片段转为向量并存入向量数据库

## 第二阶段：回答阶段
3. **召回 (Retrieval)** - 根据问题找到最相关的片段
4. **重排 (Re-rank)** - 用更精确的模型重新排序
5. **生成 (Generation)** - 用 LLM 生成最终答案

---
## 第一部分：导入依赖库

In [ ]:
import os
import re
from typing import List, Tuple, Optional
from dataclasses import dataclass

# 向量数据库
import chromadb
from chromadb.config import Settings

# Embedding 模型（将文本转为向量）
from sentence_transformers import SentenceTransformer

# Re-rank 重排序模型
from sentence_transformers import CrossEncoder

# Google Gemini API（用于最终答案生成）
import google.generativeai as genai

print("✅ 所有依赖已导入")

: 

---
## 第二部分：数据结构定义

In [ ]:
@dataclass
class TextChunk:
    """
    文本片段的数据结构
    
    Attributes:
        content: 片段的文本内容
        source: 来源文件名
        chunk_id: 片段唯一标识
        metadata: 其他元数据（如章节、页码等）
    """
    content: str
    source: str
    chunk_id: int
    metadata: dict = None


@dataclass
class SearchResult:
    """
    搜索结果的数据结构
    
    Attributes:
        chunk: 文本片段
        score: 相似度分数（越高越相关）
        rank: 重排后的排名
    """
    chunk: TextChunk
    score: float
    rank: int = 0

print("✅ 数据结构定义完成")

---
## 第三部分：分片器 (Chunker)

将长文档切分成适合检索的小片段。

**分片策略的选择直接影响检索效果：**
- 太小：片段缺乏完整语义
- 太大：检索精度下降，包含太多无关内容

**常用策略：**
1. 固定长度分片：按字符数切分
2. 语义分片：按段落、句子切分
3. 滑动窗口：允许片段之间有重叠

In [ ]:
class TextChunker:
    """文本分片器"""

    def __init__(
        self,
        chunk_size: int = 500,      # 每个片段的最大字符数
        overlap: int = 50,          # 片段之间的重叠字符数
        min_chunk_size: int = 100   # 最小片段长度
    ):
        """
        Args:
            chunk_size: 目标片段大小，500 字符约 150-200 个中文字
            overlap: 重叠大小，帮助保留跨片段的语义
            min_chunk_size: 过滤过短的片段
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.min_chunk_size = min_chunk_size

    def chunk_by_paragraph(self, text: str, source: str = "unknown") -> List[TextChunk]:
        """按段落分片（推荐用于结构化文档）"""
        paragraphs = re.split(r'\n\s*\n', text.strip())
        
        chunks = []
        chunk_id = 0

        for para in paragraphs:
            para = para.strip()
            if len(para) < self.min_chunk_size:
                continue

            if len(para) > self.chunk_size:
                sub_chunks = self._split_long_paragraph(para)
                for sub in sub_chunks:
                    chunks.append(TextChunk(content=sub, source=source, chunk_id=chunk_id))
                    chunk_id += 1
            else:
                chunks.append(TextChunk(content=para, source=source, chunk_id=chunk_id))
                chunk_id += 1

        return chunks

    def chunk_by_fixed_size(self, text: str, source: str = "unknown") -> List[TextChunk]:
        """按固定长度分片（推荐用于无结构文本）"""
        chunks = []
        chunk_id = 0
        start = 0

        while start < len(text):
            end = start + self.chunk_size
            chunk_text = text[start:end].strip()

            if len(chunk_text) >= self.min_chunk_size:
                chunks.append(TextChunk(content=chunk_text, source=source, chunk_id=chunk_id))
                chunk_id += 1

            start += self.chunk_size - self.overlap

        return chunks

    def _split_long_paragraph(self, text: str) -> List[str]:
        """将过长段落按句子切分"""
        sentences = re.split(r'[。！？.!?]\s*', text)
        
        sub_chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= self.chunk_size:
                current_chunk += sentence + "。"
            else:
                if current_chunk:
                    sub_chunks.append(current_chunk.strip())
                current_chunk = sentence + "。"

        if current_chunk:
            sub_chunks.append(current_chunk.strip())

        return sub_chunks

print("✅ 分片器定义完成")

### 实验：测试分片器

In [ ]:
# 创建分片器实例
chunker = TextChunker(chunk_size=200, overlap=30, min_chunk_size=50)

# 测试文本
test_text = """
Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。
Python 的设计哲学强调代码的可读性和简洁性。

机器学习是人工智能的一个分支，让计算机能够从数据中学习。
监督学习使用标注数据训练模型。

深度学习使用神经网络处理复杂模式。
"""

# 按段落分片
chunks = chunker.chunk_by_paragraph(test_text, source="test")

print(f"生成了 {len(chunks)} 个片段:\n")
for chunk in chunks:
    print(f"[片段 {chunk.chunk_id}] 长度: {len(chunk.content)}")
    print(f"内容: {chunk.content[:80]}...\n")

---
## 第四部分：向量索引器 (Indexer)

使用 Embedding 模型将文本转为向量，存入 ChromaDB。

### 什么是 Embedding？
- 将文本映射到高维向量空间（如 384 维或 768 维）
- 语义相近的文本，向量距离也近
- 可以用余弦相似度衡量文本相似度

### 常用 Embedding 模型
- `all-MiniLM-L6-v2`：英文，速度快，384 维
- `paraphrase-multilingual-MiniLM-L12-v2`：多语言，384 维

In [ ]:
class VectorIndexer:
    """向量索引器"""

    def __init__(
        self,
        model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
        collection_name: str = "knowledge_base",
        persist_directory: str = None
    ):
        """
        Args:
            model_name: Hugging Face 上的 Embedding 模型名称
            collection_name: ChromaDB 集合名称
            persist_directory: 持久化目录，None 表示内存模式
        """
        print(f"⏳ 正在加载 Embedding 模型: {model_name} ...")

        self.embedding_model = SentenceTransformer(model_name)

        if persist_directory:
            self.client = chromadb.PersistentClient(path=persist_directory)
        else:
            self.client = chromadb.EphemeralClient()

        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

        print(f"✅ 模型加载完成！向量维度: {self.embedding_model.get_sentence_embedding_dimension()}")

    def index_chunks(self, chunks: List[TextChunk]) -> int:
        """将文本片段索引到向量数据库"""
        if not chunks:
            return 0

        texts = [chunk.content for chunk in chunks]
        embeddings = self.embedding_model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        ids = [f"{chunk.source}_{chunk.chunk_id}" for chunk in chunks]
        metadatas = [
            {"source": chunk.source, "chunk_id": chunk.chunk_id}
            for chunk in chunks
        ]

        self.collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas
        )

        print(f"✅ 成功索引 {len(chunks)} 个片段")
        return len(chunks)

    def search(self, query: str, top_k: int = 10) -> List[Tuple[str, float, dict]]:
        """向量相似度搜索"""
        query_embedding = self.embedding_model.encode(query)

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "distances", "metadatas"]
        )

        search_results = []
        for i in range(len(results['documents'][0])):
            distance = results['distances'][0][i]
            similarity = 1 - distance  # 余弦相似度
            search_results.append((
                results['documents'][0][i],
                similarity,
                results['metadatas'][0][i]
            ))

        return search_results

print("✅ 向量索引器定义完成")

### 实验：测试向量索引

In [ ]:
# 初始化索引器（内存模式）
indexer = VectorIndexer()

# 索引片段
indexer.index_chunks(chunks)

# 测试搜索
query = "什么是机器学习？"
results = indexer.search(query, top_k=3)

print(f"\n查询: {query}\n")
for text, score, meta in results:
    print(f"[相似度: {score:.4f}] {text[:80]}...\n")

---
## 第五部分：重排器 (Re-ranker)

### 为什么需要重排？
- 向量检索基于近似搜索，可能有误差
- CrossEncoder 可以同时考虑 Query 和 Document 的交互
- 精度更高，但速度较慢

### 工作原理
1. **向量检索**：Query 和 Document 分别编码，再计算相似度
2. **CrossEncoder**：将 Query + Document 一起输入模型，直接输出相关性分数

In [ ]:
class ReRanker:
    """重排器"""

    def __init__(self, model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-X2"):
        print(f"⏳ 正在加载重排模型: {model_name} ...")
        self.reranker = CrossEncoder(model_name)
        print("✅ 重排模型加载完成！")

    def rerank(
        self,
        query: str,
        candidates: List[Tuple[str, float, dict]],
        top_k: int = 5
    ) -> List[SearchResult]:
        """对候选结果重新排序"""
        if not candidates:
            return []

        pairs = [(query, text) for text, _, _ in candidates]
        scores = self.reranker.predict(pairs)

        indexed_scores = list(enumerate(scores))
        indexed_scores.sort(key=lambda x: x[1], reverse=True)

        results = []
        for rank, (idx, score) in enumerate(indexed_scores[:top_k]):
            text, _, metadata = candidates[idx]
            results.append(SearchResult(
                chunk=TextChunk(
                    content=text,
                    source=metadata.get("source", "unknown"),
                    chunk_id=metadata.get("chunk_id", 0)
                ),
                score=float(score),
                rank=rank + 1
            ))

        return results

print("✅ 重排器定义完成")

### 实验：测试重排

In [ ]:
# 初始化重排器
reranker = ReRanker()

# 重排结果
ranked = reranker.rerank(query, results, top_k=3)

print(f"\n重排后的结果:\n")
for r in ranked:
    print(f"[排名 {r.rank}] 分数: {r.score:.4f}")
    print(f"内容: {r.chunk.content[:80]}...\n")

---
## 第六部分：生成器 (Generator)

将检索到的相关片段和用户问题一起喂给 LLM，生成最终答案。

### 为什么需要 RAG 而不是直接问 LLM？
- LLM 知识有截止日期，无法获取最新信息
- LLM 不知道你的私有数据
- RAG 可以让 LLM 基于具体上下文回答，减少幻觉

In [ ]:
class AnswerGenerator:
    """答案生成器"""

    def __init__(self, api_key: str, model_name: str = "gemini-1.5-flash"):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name)
        print(f"✅ Gemini 模型已初始化: {model_name}")

    def generate(
        self,
        query: str,
        context_chunks: List[SearchResult],
        max_context_length: int = 3000
    ) -> str:
        """生成答案"""
        context = self._build_context(context_chunks, max_context_length)
        prompt = self._build_prompt(query, context)
        response = self.model.generate_content(prompt)
        return response.text

    def _build_context(self, chunks: List[SearchResult], max_length: int) -> str:
        context_parts = []
        total_length = 0

        for result in chunks:
            chunk_text = f"【参考资料 {result.rank}】\n{result.chunk.content}\n"
            if total_length + len(chunk_text) > max_length:
                break
            context_parts.append(chunk_text)
            total_length += len(chunk_text)

        return "\n".join(context_parts)

    def _build_prompt(self, query: str, context: str) -> str:
        return f"""你是一个知识问答助手。请根据以下参考资料回答用户的问题。

要求：
1. 优先使用参考资料中的信息回答
2. 如果参考资料中没有相关信息，请明确说明
3. 回答要简洁准确，不要编造信息
4. 可以引用参考资料的编号（如【参考资料 1】）

---

{context}

---

用户问题：{query}

请回答："""

print("✅ 生成器定义完成")

---
## 第七部分：RAG 系统 - 整合所有组件

In [ ]:
class RAGSystem:
    """RAG 系统主类"""

    def __init__(
        self,
        embedding_model: str = "paraphrase-multilingual-MiniLM-L12-v2",
        reranker_model: str = "cross-encoder/mmarco-mMiniLMv2-L12-X2",
        gemini_api_key: str = None,
        persist_directory: str = None
    ):
        print("\n" + "="*60)
        print("🚀 初始化 RAG 系统")
        print("="*60 + "\n")

        self.chunker = TextChunker(chunk_size=500, overlap=50)
        self.indexer = VectorIndexer(
            model_name=embedding_model,
            persist_directory=persist_directory
        )
        self.reranker = ReRanker(model_name=reranker_model)

        if gemini_api_key:
            self.generator = AnswerGenerator(api_key=gemini_api_key)
        else:
            self.generator = None
            print("⚠️ 未提供 Gemini API Key，生成功能将不可用")

        print("\n" + "="*60)
        print("✅ RAG 系统初始化完成！")
        print("="*60 + "\n")

    def ingest_text(self, text: str, source: str = "direct_input", chunk_method: str = "paragraph") -> int:
        """摄入文本"""
        if chunk_method == "paragraph":
            chunks = self.chunker.chunk_by_paragraph(text, source)
        else:
            chunks = self.chunker.chunk_by_fixed_size(text, source)

        print(f"生成了 {len(chunks)} 个片段")
        return self.indexer.index_chunks(chunks)

    def query(
        self,
        question: str,
        top_k_retrieval: int = 10,
        top_k_rerank: int = 3,
        return_context_only: bool = False
    ) -> str:
        """回答问题"""
        print("\n" + "-"*40)
        print(f"❓ 用户问题: {question}")
        print("-"*40)

        # Step 1: 召回
        print("🔍 Step 1: 向量召回...")
        candidates = self.indexer.search(question, top_k=top_k_retrieval)
        print(f"   召回 {len(candidates)} 个候选片段")

        # Step 2: 重排
        print("📊 Step 2: 重排序...")
        ranked_results = self.reranker.rerank(question, candidates, top_k=top_k_rerank)
        print(f"   保留前 {len(ranked_results)} 个最相关片段")

        for result in ranked_results:
            print(f"   [{result.rank}] 分数: {result.score:.4f} | 来源: {result.chunk.source}")

        # Step 3: 生成
        if return_context_only or not self.generator:
            print("\n📝 返回上下文（不生成答案）")
            return ranked_results

        print("🤖 Step 3: 生成答案...")
        answer = self.generator.generate(question, ranked_results)

        print("\n" + "="*40)
        print("💬 最终答案:")
        print("="*40)
        print(answer)

        return answer

print("✅ RAG 系统定义完成")

---
## 第八部分：完整示例

In [ ]:
# 示例知识库
sample_knowledge = """
# Python 编程基础

Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。
Python 的设计哲学强调代码的可读性和简洁性。

## 变量与数据类型

Python 支持多种数据类型：
- int：整数，如 42
- float：浮点数，如 3.14
- str：字符串，如 "Hello"
- bool：布尔值，True 或 False
- list：列表，如 [1, 2, 3]
- dict：字典，如 {"name": "Alice"}

## 函数定义

使用 def 关键字定义函数：

def greet(name):
    return f"Hello, {name}!"

# 机器学习入门

机器学习是人工智能的一个分支，让计算机能够从数据中学习。

## 监督学习

监督学习使用标注数据训练模型。常见任务包括：
- 分类：预测类别标签
- 回归：预测连续数值

## 无监督学习

无监督学习使用未标注数据。常见任务包括：
- 聚类：将相似数据分组
- 降维：减少特征数量
"""

In [ ]:
# 初始化 RAG 系统（不需要 API Key 也可以测试检索）
rag = RAGSystem(gemini_api_key=None)

In [ ]:
# 摄入知识库
rag.ingest_text(sample_knowledge, source="programming_knowledge")

In [ ]:
# 测试查询 1
results = rag.query("Python 支持哪些数据类型？", return_context_only=True)

In [ ]:
# 测试查询 2
results = rag.query("什么是监督学习？", return_context_only=True)

---
## 使用 Gemini API 生成答案

如果你有 Gemini API Key，可以取消下面的注释来测试完整流程。

In [ ]:
# 设置你的 API Key
# API_KEY = "your-gemini-api-key"
# 
# rag_with_gen = RAGSystem(gemini_api_key=API_KEY)
# rag_with_gen.ingest_text(sample_knowledge, source="programming_knowledge")
# rag_with_gen.query("Python 支持哪些数据类型？")

---
## 总结

### RAG 流程回顾

1. **准备阶段**
   - 分片：`chunker.chunk_by_paragraph()` 或 `chunk_by_fixed_size()`
   - 索引：`indexer.index_chunks()`

2. **回答阶段**
   - 召回：`indexer.search()`
   - 重排：`reranker.rerank()`
   - 生成：`generator.generate()`

### 进一步学习

- 尝试不同的分片策略和参数
- 使用自己的文档测试
- 探索不同的 Embedding 和重排模型
- 添加更多元数据（如章节、页码）用于过滤